In [ ]:
# !pip install --upgrade transformers

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Dùng GPU để train model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng: {device}")

Sử dụng: cuda


In [ ]:
# ---- ĐỌC DATA -----
raw_url = 'https://github.com/thethien8a/Sentiment-Analysis-Vietnamese/raw/main/comment_data/Tram_dung_chan_Jack_shuffled.xlsx'
df_cmt = pd.read_excel(raw_url)
df_cmt = df_cmt[['textDisplay', 'label']]
df_cmt.head()

,textDisplay,label
0,Đừng dối lòng phủ nhận tài năng của Jack. Nhìn...,2.0
1,mình là tài xế chạy grab gần 8 năm chưa bao gi...,2.0
2,Sky qua ủng hộ Jack \nChúc dự án này thành côn...,2.0
3,"Không uổn công tui cày từ 5 giờ sáng tới giờ, ...",2.0
4,Đa số nghệ sĩ trong showbiz hiện giờ á toàn qu...,2.0


## Tạo từ điển viết tắt và Tạo hàm

In [ ]:
TEENCODE_DICT = {
    'ko': 'không', 'k': 'không', 'kh': 'không', 'bt': 'biết',
    'dc': 'được', 'đc': 'được', "ak":"à", "àh":"à",
    "uk":"ừ","uh":"ừ", "cmnl":"con mẹ nó luôn",
    'ad': 'admin', 'ntn': 'như thế nào',
    'v': 'vậy', 'r': 'rồi', "qtr":"quá trời",
    'cx': 'cũng',
    'ib': 'inbox',
    'rep': 'trả lời',
    'mn': 'mọi người',
    "goy":"rồi","xl":"xin lỗi",
    "j":"gì",
    "gòy":"rồi",
    'cmt': 'comment',
    'b': 'bạn',
    "tr":"trời",
    "hk":"không",
    "tt":"truyền thông",
    'e': 'em',
    'a': 'anh',
    "ng":"người", "lm":"làm", "cầu đặc": "đầu cặc", "kẹc":"cặc", "đ":"đéo", "gđ":"gia đình", "cằk":"cặc","cặk":"cặc", 'okela':'ok', "djt":"địt",
    "trây":"jack",  "bn":"bao nhiêu","bnh":"bao nhiêu", "cmm":"con mẹ","cm":"chúng mày","clm":"cái lồn má", 'vc':'vãi cả', 'cc':'cục cứt', 'loz':'lồn', 'l':'lồn', "lol":"lồn","lòn":"lồn", "lôn":"lồn",
    'vl':'vãi lồn',"vcl":"vãi cả lồn","vkl":"vãi cả lồn", "đcm":"địt con mẹ","đm":"địt mẹ","đcmm":"địt con mẹ","đcmmm":"địt con mẹ", "đb":"đầu buồi",
    'deo':'đéo','dell':'đéo', 'vcc':"vãi cả cứt", "occho":"óc chó", "t":"tôi", "chx":"chưa", "xg":"xong",
    "vcd":"vãi cả đái","vcđ":"vãi cả đái",
    "ae":"anh em", "j":"gì",
    "soll":"son","sol":"son",
    "vaiz":"vãi",
    "mn":"mọi người","mng":"mọi người",
    "bth":"bình thường",
    "m":"mày",
    "cm":"chúng mày",
    "mn":"mọi người",
    "st":"Sơn Tùng",
    "ch":"chưa",
    "vs":"với",
    "sv":"súc vật",
    "c":"cứt",
    "t.o.p":"top","to.p":"top",
    "ncc":"như cục cứt",
    "dit":"địt","djt":"địt",
    "hth":"hiếu thứ hai",
    "qc":"quảng cáo",
    "vn":"việt nam", "vaiz":"vãi",
    "r":"rồi", "nge":"nghe", "tiktok":"tik tok", "ytb":"youtube", "fb":"facebook", "vcut":"vãi cứt","shit":"cứt", "fl":"follow"
}

EMOJI_DICT = {
    "😢":" bieu_tuong_buon ",
    "🎉":" bieu_tuong_chuc_mung ",
    "❤":" bieu_tuong_trai_tim ",
    "🤣":" bieu_tuong_mat_cuoi ",
    "😂":" bieu_tuong_mat_cuoi ",
    "😭":" bieu_tuong_khoc ",
    "🔥":" bieu_tuong_chay ",
    "💩":" bieu_tuong_cut ",
    "🥰":" bieu_tuong_yeu_thuong ",
    "😮":" bieu_tuong_ngac_nhien ",
    "😅":" bieu_tuong_lo_lang ",
    "<3":" bieu_tuong_trai_tim "
}

In [ ]:
def preprocessing_basic(text):
  text = re.sub(r'http\S+','',text)
  text = re.sub(r'<.*?>', '', text)
  text = text.strip()
  text = text.replace("t o p","top").replace("tốp","top")
  text = text.replace("T o p","Top").replace("T O P","TOP")
  text = text.replace("v i e w","view").replace("c à y","cày")
  text = text.replace("V I E W", "VIEW")
  text = text.replace("đốn lòm","đóm lồn").replace("đốm lòn","đóm lồn")
  return text

def replace_teencode(text,teencode_dict):
  words = text.split()
  for i,word in enumerate(words):
    word_lower = word.lower()
    if (word_lower in teencode_dict):
      words[i] = teencode_dict[word_lower]
  return " ".join(words)

def replace_emoji(text,emoji_dict):
  for emoji, word in emoji_dict.items():
      text = text.replace(emoji, word)
  text = re.sub(r'\s+', ' ', text).strip()
  return text


def keep_chars(text):
  vietnamese_chars_lowercase = 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ'
  vietnamese_chars_uppercase = vietnamese_chars_lowercase.upper()
  allowed_chars = vietnamese_chars_lowercase + 'a-zA-Z1-9_' + vietnamese_chars_uppercase
  unwanted_chars_pattern = re.compile(f'[^{allowed_chars}\s?!.,()]')
  text = unwanted_chars_pattern.sub('', text)
  text = re.sub(r'\s+',' ',text)
  text = text.strip()
  return text

def preprocess_comment(text):
    global TEENCODE_DICT
    global EMOJI_DICT
    global STOP_WORDS

    text = preprocessing_basic(text)
    text = replace_teencode(text, TEENCODE_DICT)
    text = replace_emoji(text,EMOJI_DICT)
    text = keep_chars(text)

    return text

## Tiền xử lý

In [ ]:
# Xoá NaN
df_cmt = df_cmt.dropna()

# Lọc trùng bình luận (tránh seeding)
df_cmt = df_cmt.drop_duplicates(subset=['textDisplay'], keep='first')
df_cmt['label'] = df_cmt['label'].astype(int)
df_cmt["textDisplay"] = df_cmt["textDisplay"].fillna("").apply(preprocess_comment)

In [ ]:
df_cmt.head(100)

,textDisplay,label
0,Đừng dối lòng phủ nhận tài năng của Jack. Nhìn...,2
1,mình là tài xế chạy grab gần 8 năm chưa bao gi...,2
2,Sky qua ủng hộ Jack Chúc dự án này thành công ...,2
3,"Không uổn công tui cày từ 5 giờ sáng tới giờ, ...",2
4,Đa số nghệ sĩ trong showbiz hiện giờ á toàn qu...,2
...,...,...
95,"Thấy ảnh bị chèn ép mà thấy thương luôn á tr, ...",2
96,không thích cách sống j. Nhưng nhạc hay thì ph...,2
97,fan Sơn Tùng ung ho JACK,2
98,Tao Đóm Chúa. tôi sinh 1241997. tôi đủ 28t và ...,2


In [ ]:
df_cmt.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3899 entries, 0 to 4227
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   textDisplay  3899 non-null   object
 1   label        3899 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 91.4+ KB


In [ ]:
# Train_test_split

# Tách tập train và test
comments = df_cmt["textDisplay"]
y = df_cmt["label"]
train_texts, test_texts, train_labels, test_labels = train_test_split(
    comments, y, test_size=0.2, stratify=y, random_state=42
)

# Ta sử dụng thêm 1 tập nữa là validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, stratify=train_labels, random_state=42
)


In [ ]:
# Chúng ta tiếp tục chuyển thành kiểu dữ liệu phù hợp với input của Huggingface
train_dataset = Dataset.from_dict({
    'text': train_texts,
    'labels': train_labels
})

val_dataset = Dataset.from_dict({
    'text': val_texts,
    'labels': val_labels
})

test_dataset = Dataset.from_dict({
    'text': test_texts,
    'labels': test_labels
})

In [ ]:
# Chọn model từ huggingface (trong TH này tôi sử dụng vietnamese sentiment)
model_name = "wonrax/phobert-base-vietnamese-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=256
    )

In [ ]:
# Tokenize datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2495 [00:00<?, ? examples/s]

Map:   0%|          | 0/624 [00:00<?, ? examples/s]

Map:   0%|          | 0/780 [00:00<?, ? examples/s]

In [ ]:
# Xem thử mẫu đầu tiên
print(tokenized_train[0])

{'text': 'Nhớ xem hết quảng cáo cho Jack nhé các bạn', 'labels': 2, 'input_ids': [0, 6654, 305, 351, 33504, 9866, 13, 18012, 2083, 9, 88, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

## Sử dụng model pre-train dự đoán thử để xem kết quả

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [ ]:
def predict_with_model(model, tokenized_dataset, dataset_name="Test"):
    """
    Evaluate pre-trained model trên dataset đã tokenize
    """
    model.eval()
    all_predictions = []

    with torch.no_grad():
        for i in tqdm(range(len(tokenized_dataset)), desc=f"Predicting {dataset_name}"):
            # Get single example
            example = tokenized_dataset[i]

            # Prepare input
            inputs = {
                'input_ids': torch.tensor([example['input_ids']]).to(device),
                'attention_mask': torch.tensor([example['attention_mask']]).to(device)
            }

            # Predict
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)
            all_predictions.append(predictions.cpu().item())
    return all_predictions

def evaluate_model(all_predictions, true_labels):
    f1_weighted = f1_score(true_labels, all_predictions, average='weighted')
    f1_macro = f1_score(true_labels, all_predictions, average='macro')
    print(f"\n Results:")
    print(f"F1-Weighted: {f1_weighted:.4f}")
    print(f"F1-Macro:    {f1_macro:.4f}")

    print(f"\n Classification Report:")
    print(classification_report(
        true_labels,
        all_predictions,
        target_names=['Negative (0)', 'Neutral (1)', 'Positive (2)'],
        digits=4
    ))

In [ ]:
## Test trên tập dữ liệu của chúng ta
labels_predicts = predict_with_model(
    model,
    tokenized_test,
    "Test"
)

Predicting Test: 100%|██████████| 780/780 [00:12<00:00, 61.84it/s]


In [ ]:
# Đánh giá model
evaluate_model(labels_predicts,test_labels)


 Results:
F1-Weighted: 0.3460
F1-Macro:    0.3541

 Classification Report:
              precision    recall  f1-score   support

Negative (0)     0.4549    0.5663    0.5045       196
 Neutral (1)     0.1867    0.3503    0.2436       177
Positive (2)     0.4706    0.2359    0.3142       407

    accuracy                         0.3449       780
   macro avg     0.3708    0.3842    0.3541       780
weighted avg     0.4022    0.3449    0.3460       780



-> Các chỉ số như F1-Weighted, F1-Macro có vẻ kém hơn so với mô hình baseline

## Huấn luyện

In [ ]:
from sklearn.metrics import accuracy_score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    f1_weighted = f1_score(labels, predictions, average='weighted')
    f1_macro = f1_score(labels, predictions, average='macro')
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro
    }

In [ ]:
# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,  # 0: negative, 1: neutral, 2: positive
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2}
)
model.to(device)
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [36]:
# Training arguments (lấy 1 vài ý tưởng của Mạnh thêm vào)
training_args = TrainingArguments(
    output_dir="./phobert_sentiment_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    seed=42,
    fp16=True,
    report_to="tensorboard"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [37]:
print("Bắt đầu finetune nè ...")
trainer.train()
print("KẾT THÚC !!!")
local_output_dir = "./phobert_sentiment_finetuned"
trainer.save_model(local_output_dir) # Nếu lưu trên VM của colab

Bắt đầu finetune nè ...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.217700,1.247904,0.682692,0.692975,0.659010
2,0.121800,1.359209,0.685897,0.693781,0.658596
3,0.230800,1.304731,0.689103,0.688497,0.648362


KẾT THÚC !!!


## Dự đoán

In [38]:
# Load fine-tuned model
model_path = "./phobert_sentiment_finetuned"
model_finetuned = AutoModelForSequenceClassification.from_pretrained(model_path)
model_finetuned.to(device)
model_finetuned.eval()
print("Fine-tuned model loaded!")

Fine-tuned model loaded!


In [39]:
## Test trên tập dữ liệu của chúng ta
labels_predicts = predict_with_model(
    model_finetuned,
    tokenized_test,
    "Test"
)

Predicting Test: 100%|██████████| 780/780 [00:15<00:00, 50.54it/s]


## Đánh giá kết quả


In [41]:
# Test set
evaluate_model(labels_predicts,test_labels)


 Results:
F1-Weighted: 0.7125
F1-Macro:    0.6826

 Classification Report:
              precision    recall  f1-score   support

Negative (0)     0.7853    0.7653    0.7752       196
 Neutral (1)     0.4495    0.5537    0.4962       177
Positive (2)     0.8140    0.7420    0.7763       407

    accuracy                         0.7051       780
   macro avg     0.6830    0.6870    0.6826       780
weighted avg     0.7241    0.7051    0.7125       780



## Diễn giải mô hình:
1. Model đạt độ chính xác 70.51% - tức là cứ 10 câu thì đoán đúng khoảng 7 câu sentiment. So với tỷ lệ đoán mò cũng ra nhãn positive là 52.18% thì mô hình đạt qua ngưỡng này.
2. **Hiệu suất theo từng loại cảm xúc**:
+ Negative: 1 câu 78% là thực sự tiêu cực khi model nói nó là tiêu cực. Tỷ lệ phát hiện ra được các câu tiêu cực thực sự là 76%
+ Positve: ... tương tự
+ Neutral: ...

### So sánh 2 model:
1. Khi sử dụng cả 2 model (Model phobert và machine learning algorithm) thì cả 2 đều cho ta thấy được rằng lớp 1 (Neutral) đều có các chỉ số như F1-Score, precision, recall thấp
2. Nhìn chung, cả 2 mô hình đều có F1-Weighted là tương đương nhau (~ 0.7) cho thấy khả năng cân bằng giữa precision và recall thuộc dạng trung bình và đồng đều nhau.
3. Hai mô hình là tương đương nhau. Có vẻ không có cái nào vượt trội hơn cái nào. Tuy nhiên, do tập dữ liệu dành cho train model "Phobert" kém hơn model kia khoảng 300-400 dữ liệu (do tôi chia cả validation) thế nhưng mà kết quả lại ngang bằng so với machine learning model thì cho ta thấy tiềm năng phát triển cho mô hình Phobert.

#### **Kết luận**: cả 2 model hiện tại ngang cơ nhau. Nhưng model phobert có tiềm năng phát triển nếu ta cung cấp thêm dữ liệu

## Đề xuất hành động
1. Thu thập thêm dữ liệu neutral
2. Tinh chỉnh siêu tham số (nếu có thể, với khả năng hiện tại tôi không thể 😓)